In [ ]:
#load data into dataframe=df

In [ ]:

import os
import time
import pandas as pd
import requests
import re

CHECKPOINT_DIR = "path-to-save-output-file"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "llmaa_checkpoint.csv")

print(f"Checkpoint will be saved to: {CHECKPOINT_PATH}")


OPENROUTER_API_KEY = "sk----"

# Pin the dated snapshot, not the floating alias — makes results reproducible for the methods section
MODEL = "meta-llama/llama-3.1-70b-instruct"

TOPICS = [

    "Captivity, Zoos & Ethical Wildlife Management",
    "Climate Change Impact on Wildlife",
    "Habitat Loss, Deforestation & Ecosystem Degradation",
    "Wildlife Rehabilitation & Rescue",
    "Wildlife Awareness, Media & Activism",
    "Human-Wildlife Conflict",
    "Marine & Ocean Conservation",
    "Endangered Species & Extinction Risk",
    "Wildlife Crime, Poaching & Enforcement",
    "Conservation Policy, Law & Funding",
    "Indigenous & Community-Based Conservation",
    "National Parks & Protected Areas",
    "UNASSIGNED",  # escape valve — required, or every ambiguous post is force-fit
]


SYSTEM_PROMPT = """
You are an expert annotator classifying social media messages
about wildlife, biodiversity, nature, and conservation.

Most messages are written in Arabic, including Modern Standard
Arabic (MSA), Gulf, Egyptian, Levantine, and Maghrebi dialects.
Some messages may contain English, code-switching, informal
spelling, or Arabizi (Arabic transliterated into Latin letters
and numbers, such as "7" for ح and "3" for ع).

Classify the meaning of each message regardless of its dialect,
script, or writing style. Do not classify a message as UNASSIGNED
merely because it contains unfamiliar dialectal vocabulary,
informal expressions, or spelling variations.

Your task is to assign each message to exactly ONE of the
following 12 predefined topics based on its primary meaning.

Each topic has an English and Arabic definition. Both definitions
describe the same topic and should be interpreted together.

TOPIC DEFINITIONS

1. Captivity, Zoos & Ethical Wildlife Management

Wild animals kept in zoos, aquariums, breeding facilities,
or captivity, including animal welfare, living conditions,
captive breeding, management, and ethical treatment.

الحيوانات البرية الموجودة في حدائق الحيوان أو الأحواض المائية
أو مراكز التكاثر أو الأسر، بما في ذلك رفاهية الحيوانات
وظروف الاحتجاز والتكاثر في الأسر والإدارة والمعاملة الأخلاقية.


2. Climate Change Impact on Wildlife

Effects of climate change, global warming, extreme heat,
climate-related drought, changing weather patterns, and
other climatic changes on wild animals, their habitats,
migration, reproduction, and survival.

تأثير تغير المناخ والاحتباس الحراري وارتفاع درجات الحرارة
والجفاف المرتبط بالمناخ وتغير أنماط الطقس على الحيوانات
البرية وموائلها وهجرتها وتكاثرها وبقائها.


3. Habitat Loss, Deforestation & Ecosystem Degradation

Destruction, fragmentation, or degradation of natural
habitats and ecosystems affecting wildlife, including
deforestation, land conversion, urban expansion,
terrestrial pollution, and loss of biodiversity.

تدمير الموائل الطبيعية وتجزئتها وتدهور النظم البيئية
التي تعتمد عليها الحياة البرية، بما في ذلك إزالة الغابات
وتغيير استخدام الأراضي والتوسع العمراني والتلوث البري
وفقدان التنوع البيولوجي.


4. Wildlife Rehabilitation & Rescue

Rescue, veterinary treatment, rehabilitation, recovery,
and release of individual wild animals that are injured,
stranded, orphaned, trapped, or in distress.

إنقاذ الحيوانات البرية المصابة أو العالقة أو اليتيمة
أو المعرضة للخطر، وعلاجها بيطرياً وإعادة تأهيلها
وتعافيها وإطلاقها مجدداً في البرية.


5. Wildlife Awareness, Media & Activism

Public education, awareness campaigns, environmental
advocacy, wildlife documentaries, photography, observation,
species information, animal behavior, and general
appreciation or promotion of wildlife and nature.

التوعية العامة والتثقيف البيئي وحملات حماية الحياة البرية
والنشاط البيئي والأفلام الوثائقية وتصوير الحيوانات
ومشاهدتها والمعلومات عن الأنواع وسلوك الحيوانات
والتقدير العام للحياة البرية والطبيعة.


6. Human-Wildlife Conflict

Harmful or disputed interactions between wild animals
and humans, including attacks on people, crop damage,
livestock predation, retaliatory killing, property
damage, and efforts to improve human-wildlife coexistence.

الصراعات والتفاعلات الضارة بين الإنسان والحيوانات البرية،
بما في ذلك الهجمات على البشر وإتلاف المحاصيل وافتراس
الماشية والقتل الانتقامي والأضرار بالممتلكات والجهود
المبذولة لتحسين التعايش بين الإنسان والحياة البرية.


7. Marine & Ocean Conservation

Protection and management of marine wildlife and aquatic
ecosystems, including whales, dolphins, sea turtles,
coral reefs, marine pollution, plastic waste, overfishing,
bycatch, and threats to marine biodiversity.

حماية وإدارة الحياة البرية البحرية والنظم البيئية المائية،
بما في ذلك الحيتان والدلافين والسلاحف البحرية والشعاب
المرجانية والتلوث البحري والنفايات البلاستيكية والصيد
الجائر والصيد العرضي والتهديدات للتنوع البيولوجي البحري.


8. Endangered Species & Extinction Risk

Conservation status and survival of wildlife species,
including endangered and threatened species, declining
populations, extinction risk, species-level recovery,
and protection or reintroduction of threatened species.

حالة حفظ الأنواع البرية وبقائها، بما في ذلك الأنواع
المهددة بالانقراض وتناقص أعدادها وخطر انقراضها
وبرامج استعادة الأنواع وحمايتها وإعادة توطينها.


9. Wildlife Crime, Poaching & Enforcement

Illegal hunting, killing, capturing, trading, or
trafficking of wild animals and wildlife products,
including poaching, illegal wildlife trade, investigations,
seizures, arrests, and enforcement against wildlife crime.

الصيد غير القانوني وقتل الحيوانات البرية وأسرها
والاتجار غير المشروع بها أو بمنتجاتها، بما في ذلك
الصيد الجائر وتهريب الحيوانات والتحقيقات والمصادرات
والاعتقالات وإنفاذ قوانين مكافحة جرائم الحياة البرية.


10. Conservation Policy, Law & Funding

Government policies, legislation, regulations, funding,
donations, institutional programs, and activities by
conservation organizations intended to protect wildlife,
natural resources, and biodiversity.

السياسات الحكومية والتشريعات واللوائح والتمويل
والتبرعات والبرامج المؤسسية وأنشطة منظمات الحفاظ
على البيئة الهادفة إلى حماية الحياة البرية والموارد
الطبيعية والتنوع البيولوجي.


11. Indigenous & Community-Based Conservation

Wildlife conservation involving indigenous peoples
or local communities, including traditional ecological
knowledge, community-led protection, local natural
resource management, and land rights connected to
biodiversity conservation.

حماية الحياة البرية بمشاركة السكان الأصليين أو المجتمعات
المحلية، بما في ذلك المعرفة البيئية التقليدية ومبادرات
الحماية المجتمعية وإدارة الموارد الطبيعية محلياً
وحقوق الأراضي المرتبطة بالحفاظ على التنوع البيولوجي.


12. National Parks & Protected Areas

Establishment, management, conservation, and visitation
of national parks, nature reserves, wildlife sanctuaries,
and designated protected areas, including their ecological
importance and role in protecting wildlife.

إنشاء وإدارة وحماية وزيارة الحدائق الوطنية والمحميات
الطبيعية ومحميات الحياة البرية والمناطق المحمية،
بما في ذلك أهميتها البيئية ودورها في حماية الحيوانات البرية.


CLASSIFICATION RULES

1. PRIMARY MEANING
Identify the primary subject of the message based on its
overall meaning. Do not classify based solely on the
presence of individual keywords.

2. SINGLE TOPIC
Assign exactly ONE topic to each message. Do not return
multiple topics or combine category names.

3. SPECIFICITY
When multiple topics are relevant, select the topic that
best represents the main focus of the message.

Prefer a specific conservation issue or event over a
general topic when the specific issue is clearly the
primary focus.

4. GENERAL WILDLIFE CONTENT
Messages about wildlife education, animal behavior,
species facts, nature appreciation, wildlife photography,
documentaries, and wildlife observation can be classified
as Wildlife Awareness, Media & Activism when no more
specific topic applies.

An explicit mention of conservation or environmental
protection is not required for this category.

However, do not use this category as a fallback for
unrelated, unclear, or insufficient content.

5. OVERLAPPING TOPICS
Distinguish between categories based on the main issue
being discussed.

Examples:

- Illegal wildlife trafficking or poaching belongs to
  Wildlife Crime, Poaching & Enforcement.

- The living conditions of animals in zoos belong to
  Captivity, Zoos & Ethical Wildlife Management.

- Rescue and veterinary treatment of an individual
  injured animal belong to Wildlife Rehabilitation & Rescue.

- Species-level population decline or extinction risk
  belongs to Endangered Species & Extinction Risk.

- Wildlife attacks on people or livestock belong to
  Human-Wildlife Conflict.

- Habitat destruction caused by deforestation or
  land conversion belongs to Habitat Loss, Deforestation
  & Ecosystem Degradation.

- Wildlife impacts explicitly attributed to climate
  change belong to Climate Change Impact on Wildlife.

- A poaching incident occurring inside a national park
  belongs to Wildlife Crime, Poaching & Enforcement
  when the incident is the primary focus.

- Community participation alone is insufficient for
  Indigenous & Community-Based Conservation unless
  the message concerns a community role in conservation.

- Marine wildlife and marine ecosystem protection
  belong to Marine & Ocean Conservation when the
  marine conservation issue is the primary focus.

6. CONTEXTUAL INTERPRETATION
Interpret dialectal expressions, informal language,
code-switching, and Arabizi based on their intended
meaning rather than exact vocabulary.

Do not assume that a message is unrelated simply
because its wording differs from the definitions.

7. INSUFFICIENT EVIDENCE
Do not infer a specific conservation event, crime,
policy, or environmental threat without sufficient
support from the message.

For example, mentioning a national park is not
automatically evidence of poaching, and mentioning
an endangered animal does not necessarily mean that
the message discusses extinction risk.

8. UNASSIGNED
Return UNASSIGNED only when one of the following applies:

- The message does not reasonably fit any of the
  12 predefined topics.

- The message is unrelated to wildlife, biodiversity,
  nature, or conservation.

- The message contains too little meaningful information
  to identify a relevant topic.

- The message is too ambiguous to determine its
  primary topic, even after considering its context.

Do not force a message into an unrelated category
just to avoid UNASSIGNED.

However, do not choose UNASSIGNED merely because
the post does not explicitly mention conservation.
General wildlife-related content may still fit
Wildlife Awareness, Media & Activism.

9. CONSISTENCY
Apply the same topic definitions and classification
principles to all messages, regardless of language,
dialect, length, writing style, or emotional tone.

10. INTERNAL DECISION
Consider the message meaning, relevant definitions,
and potential category overlaps before selecting
the most appropriate label.

Do not output your reasoning.

OUTPUT REQUIREMENTS

Return ONLY the exact English topic name from the
following list:

Captivity, Zoos & Ethical Wildlife Management
Climate Change Impact on Wildlife
Habitat Loss, Deforestation & Ecosystem Degradation
Wildlife Rehabilitation & Rescue
Wildlife Awareness, Media & Activism
Human-Wildlife Conflict
Marine & Ocean Conservation
Endangered Species & Extinction Risk
Wildlife Crime, Poaching & Enforcement
Conservation Policy, Law & Funding
Indigenous & Community-Based Conservation
National Parks & Protected Areas
UNASSIGNED

Do not return explanations, translations, numbering,
quotation marks, punctuation, or additional text.
""".strip()



RETRY_LABELS = {"API_ERROR", "EMPTY_RESPONSE"}   # re-attempted on the next run
MAX_TOKENS = 300                                 # 30 is too low for reasoning models


def classify_message(message, max_retries=4):
    failure = "API_ERROR"
    problem = "no attempt made"

    for attempt in range(max_retries):
        if attempt:
            time.sleep(2 ** attempt)
        try:
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                    "Content-Type": "application/json",
                },
                json={
                    "model": MODEL,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": str(message)},
                    ],
                    "temperature": 0,
                    "max_tokens": MAX_TOKENS,
                },
                timeout=60,
            )

            if response.status_code == 429 or response.status_code >= 500:
                failure, problem = "API_ERROR", f"HTTP {response.status_code}"
                continue

            response.raise_for_status()
            data = response.json()

            # OpenRouter can return HTTP 200 with an error body and no choices
            if not data.get("choices"):
                failure, problem = "API_ERROR", f"no choices: {data.get('error')}"
                continue

            choice = data["choices"][0]
            content = (choice.get("message") or {}).get("content")

            # content is None/empty when the model spent its tokens on
            # reasoning, refused, or was cut off by a content filter
            if not isinstance(content, str) or not content.strip():
                failure = "EMPTY_RESPONSE"
                problem = f"empty content, finish_reason={choice.get('finish_reason')!r}"
                continue

            return normalize_topic(content)

        except (requests.exceptions.RequestException, ValueError) as e:
            failure, problem = "API_ERROR", str(e)

    print(f"Failed after {max_retries} attempts: {problem}")
    return failure


def normalize_for_compare(s):
    s = s.replace('&', 'and')
    s = re.sub(r'[^\w\s]', '', s)      # strip commas, periods, quotes
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s

TOPIC_LOOKUP = {normalize_for_compare(t): t for t in TOPICS}

def normalize_topic(raw):
    cleaned = raw.strip().strip('."\'').strip()
    if cleaned in TOPICS:
        return cleaned
    key = normalize_for_compare(cleaned)
    if key in TOPIC_LOOKUP:
        return TOPIC_LOOKUP[key]
    if key:
        for norm_key, original in TOPIC_LOOKUP.items():   # loose substring fallback
            if norm_key in key or key in norm_key:
                return original
    print(f"UNRECOGNIZED OUTPUT: {raw!r}")
    return "INVALID_OUTPUT"


def needs_label(value):
    # NaN covers rows reloaded from a checkpoint CSV, where None becomes NaN
    return pd.isna(value) or value in RETRY_LABELS


CHECKPOINT_EVERY = 50

if "Human_Topic" in df.columns:
    predictions = df["Human_Topic"].astype(object).tolist()
else:
    predictions = [None] * len(df)

todo = [i for i, value in enumerate(predictions) if needs_label(value)]
print(f"{len(todo):,} of {len(df):,} rows to classify")

for done, i in enumerate(todo, start=1):
    message = df["Message"].iloc[i]

    if pd.isna(message) or not str(message).strip():
        predictions[i] = "EMPTY_MESSAGE"
    else:
        predictions[i] = classify_message(message)

    if done % CHECKPOINT_EVERY == 0:
        df["Human_Topic"] = predictions
        df.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")
        print(f"Checkpointed at {done}/{len(todo)}")

df["Human_Topic"] = predictions
df.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")

# sanity checks before moving to agreement analysis
print(df["Human_Topic"].value_counts(dropna=False))
problem_labels = ["INVALID_OUTPUT", "API_ERROR", "EMPTY_RESPONSE", "EMPTY_MESSAGE"]
print("Invalid/error rows:", df["Human_Topic"].isin(problem_labels).sum())